# 📊 Análise Exploratória de Leads - Corretora Internacional

In [13]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-muted')
sns.set_palette('muted')

# Caminho para o CSV processado
df_path = Path.cwd().parents[1] / "Data" / "PROCESSED" / "leads_processados_para_pbi.csv"
df = pd.read_csv(df_path, parse_dates=["data_cadastro"])
df.head()


,lead_id,origem,data_cadastro,status_conversao,dias_ate_1o_trade,valor_deposito,perfil,pais,foi_convertido,ano_mes_cadastro
0,210428,Instagram,2025-01-07,Não Convertido,NaN,0.0,Moderado,Ukraine,False,2025-01
1,210429,E-mail Marketing,2025-01-08,Não Convertido,NaN,0.0,Agressivo,Burundi,False,2025-01
2,210430,Instagram,2025-01-07,Não Convertido,NaN,0.0,Moderado,Korea,False,2025-01
3,210431,Orgânico,2025-01-10,Não Convertido,NaN,0.0,Conservador,Turks and Caicos Islands,False,2025-01
4,210432,YouTube Ads,2025-01-11,Não Convertido,NaN,0.0,Conservador,Luxembourg,False,2025-01


## 📌 Informações Gerais

In [3]:
print("Total de linhas:", len(df))
print("Total de colunas:", len(df.columns))



Total de linhas: 267448
Total de colunas: 10


In [4]:
display("Colunas disponíveis:", list(df.columns))
print("\nTipos de dados:")
print(df.dtypes)


'Colunas disponíveis:'

['lead_id',
 'origem',
 'data_cadastro',
 'status_conversao',
 'dias_ate_1o_trade',
 'valor_deposito',
 'perfil',
 'pais',
 'foi_convertido',
 'ano_mes_cadastro']


Tipos de dados:
lead_id                       int64
origem                       object
data_cadastro        datetime64[ns]
status_conversao             object
dias_ate_1o_trade           float64
valor_deposito              float64
perfil                       object
pais                         object
foi_convertido                 bool
ano_mes_cadastro             object
dtype: object


In [5]:
print("\nValores ausentes por coluna:")
display(df.isnull().sum())


Valores ausentes por coluna:


lead_id                   0
origem                    0
data_cadastro             0
status_conversao          0
dias_ate_1o_trade    253980
valor_deposito            0
perfil                    0
pais                      0
foi_convertido            0
ano_mes_cadastro          0
dtype: int64

## 📈 Quantidade de Leads por Plataforma (Origem)

In [14]:
origem_counts = df['origem'].value_counts().reset_index()
origem_counts.columns = ['origem', 'total_leads']

fig_origem = px.bar(
    origem_counts,
    x='origem',
    y='total_leads',
    text='total_leads',
    title='📢 Leads por Origem',
    labels={'total_leads': 'Total de Leads', 'origem': 'Origem'},
)

fig_origem.update_traces(marker_color='rgb(26, 118, 255)', textposition='outside')
fig_origem.update_layout(
    xaxis_tickangle=-30,
    yaxis_title='Total de Leads',
    xaxis_title='Origem',
    uniformtext_minsize=8,
    uniformtext_mode='hide',
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='white'
)

fig_origem.show()

## 🔄 Taxa de Conversão por Origem

In [15]:

# Calcula taxa de conversão por origem
conv_df = df.groupby('origem')['status_conversao'].value_counts(normalize=True).unstack().fillna(0)
conv_df = conv_df.reset_index()
conv_df['taxa_conversao'] = conv_df['Convertido'] * 100  # em percentual

# Ordena por taxa de conversão
conv_df = conv_df.sort_values('taxa_conversao', ascending=True)

# Gráfico com Plotly
fig_conv = px.bar(
    conv_df,
    x='taxa_conversao',
    y='origem',
    orientation='h',
    text='taxa_conversao',
    title='🎯 Taxa de Conversão por Origem (%)',
    labels={'taxa_conversao': 'Taxa de Conversão (%)', 'origem': 'Origem'},
)

fig_conv.update_traces(marker_color='mediumseagreen', texttemplate='%{text:.1f}%', textposition='outside')
fig_conv.update_layout(
    xaxis=dict(title='Taxa de Conversão (%)'),
    yaxis=dict(title='Origem'),
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='white'
)

fig_conv.show()



## ⏱️ Tempo Médio até o 1º Trade por Origem

In [16]:
# Calcula média de dias até o 1º trade por origem
media_dias_trade = (
    df[df['dias_ate_1o_trade'].notnull()]
    .groupby('origem')['dias_ate_1o_trade']
    .mean()
    .sort_values()
    .reset_index()
)

# Gráfico com Plotly


fig_dias = px.bar(
    media_dias_trade,
    x='dias_ate_1o_trade',
    y='origem',
    orientation='h',
    text='dias_ate_1o_trade',
    title='⏱️ Média de Dias até o 1º Trade por Origem',
    labels={'dias_ate_1o_trade': 'Dias até o 1º Trade', 'origem': 'Origem'}
)

fig_dias.update_traces(
    marker_color='steelblue',
    texttemplate='%{text:.1f} dias',
    textposition='outside'
)
fig_dias.update_layout(
    xaxis_title='Dias',
    yaxis_title='Origem',
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='white'
)

fig_dias.show()


## 💰 Valor Médio de Depósito por Plataforma

In [17]:
# Calcula depósito médio por origem (apenas leads convertidos)
deposito_medio = (
    df[df['valor_deposito'] > 0]
    .groupby('origem')['valor_deposito']
    .mean()
    .sort_values()
    .reset_index()
)

# Gráfico interativo com Plotly


fig_deposito = px.bar(
    deposito_medio,
    x='valor_deposito',
    y='origem',
    orientation='h',
    text='valor_deposito',
    title='💰 Depósito Médio por Origem',
    labels={'valor_deposito': 'Depósito Médio (R$)', 'origem': 'Origem'}
)

fig_deposito.update_traces(
    marker_color='darkorange',
    texttemplate='R$ %{text:.2f}',
    textposition='outside'
)

fig_deposito.update_layout(
    xaxis_title='Valor Médio (R$)',
    yaxis_title='Origem',
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='white'
)

fig_deposito.show()


## 🌍 País com Maior Número de Leads Convertidos

In [9]:
import plotly.express as px

# Converter a coluna de data
df['data_cadastro'] = pd.to_datetime(df['data_cadastro'])

# Leads convertidos por país
df_convertidos = df[df['foi_convertido']]

pais_leads = df_convertidos['pais'].value_counts().nlargest(15).reset_index()
pais_leads.columns = ['pais', 'total_leads']

fig_leads = px.bar(
    pais_leads,
    x='total_leads',
    y='pais',
    orientation='h',
    title='🌍 Países com Maior Número de Leads Convertidos',
    color='total_leads',
    color_continuous_scale='Viridis',
    hover_data={'total_leads': True, 'pais': False}
)
fig_leads.update_layout(yaxis={'categoryorder': 'total ascending'})
fig_leads.show()


## 💸 País com Maior Volume de Investimento

In [10]:
investimento_pais = df_convertidos.groupby('pais')['valor_deposito'].sum().nlargest(15).reset_index()

fig_invest = px.bar(
    investimento_pais,
    x='valor_deposito',
    y='pais',
    orientation='h',
    title='💰 Países com Maior Volume Investido',
    color='valor_deposito',
    color_continuous_scale='Cividis',
    hover_data={'valor_deposito': ':.2f', 'pais': False}
)
fig_invest.update_layout(yaxis={'categoryorder': 'total ascending'})
fig_invest.show()


## 🧬 Conversão por Perfil de Trader

In [18]:
# Calcula taxa de conversão por perfil
perfil_conv = (
    df[df['perfil'].notnull()]
    .groupby('perfil')['status_conversao']
    .value_counts(normalize=True)
    .unstack()
    .fillna(0)
    .reset_index()
    .sort_values(by='Convertido')
)

# Gráfico com Plotly


fig_perfil = px.bar(
    perfil_conv,
    x='Convertido',
    y='perfil',
    orientation='h',
    title='🧠 Taxa de Conversão por Perfil',
    text='Convertido',
    labels={'Convertido': 'Taxa de Conversão (%)', 'perfil': 'Perfil'}
)

fig_perfil.update_traces(
    marker_color='mediumseagreen',
    texttemplate='%{text:.1%}',
    textposition='outside'
)

fig_perfil.update_layout(
    xaxis_title='Taxa de Conversão (%)',
    yaxis_title='Perfil do Lead',
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='white'
)

fig_perfil.show()


## 📅 Leads Cadastrados por Mês

In [19]:
# Geração da coluna mês/ano
df['mes_ano'] = df['data_cadastro'].dt.to_period('M').astype(str)

# Agrupamento por mês
leads_mes = df['mes_ano'].value_counts().sort_index().reset_index()
leads_mes.columns = ['mes_ano', 'total_leads']

# Gráfico com Plotly


fig_mes = px.bar(
    leads_mes,
    x='mes_ano',
    y='total_leads',
    title='📅 Leads por Mês',
    labels={'mes_ano': 'Mês/Ano', 'total_leads': 'Total de Leads'},
    text='total_leads'
)

fig_mes.update_traces(
    marker_color='cornflowerblue',
    textposition='outside'
)

fig_mes.update_layout(
    xaxis_tickangle=-45,
    xaxis_title='Mês',
    yaxis_title='Total de Leads',
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='white'
)

fig_mes.show()
